# CineRec — Entrenamiento y Evaluación de Modelos

Este notebook entrena y evalúa los cuatro modelos de recomendación del sistema CineRec y genera una comparativa visual de su rendimiento.

**Modelos:**
1. Popularity Recommender (baseline)
2. Collaborative Filtering — SVD
3. Content-Based — TF-IDF + Cosine Similarity
4. Classification — Random Forest

## 1. Importaciones y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_split
from surprise.accuracy import rmse

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix
)

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

BASE_DIR = Path('..') / 'data' / 'processed'

movies  = pd.read_csv(BASE_DIR / 'movies_clean.csv')
users   = pd.read_csv(BASE_DIR / 'users_clean.csv')
ratings = pd.read_csv(BASE_DIR / 'ratings_clean.csv')

movies['genres_parsed'] = movies['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

print(f'Movies:  {len(movies):,}')
print(f'Users:   {len(users):,}')
print(f'Ratings: {len(ratings):,}')

## 2. División Train / Test

Para evaluar correctamente los modelos, dividimos los datos por usuario: el 80% más antiguo de los ratings de cada usuario va al conjunto de entrenamiento y el 20% más reciente al de test. Esto simula un escenario real donde el modelo se entrena con el historial pasado y se evalúa con interacciones futuras.

In [ ]:
ratings = ratings.sort_values(['userId', 'timestamp'])

train_list, test_list = [], []

for _, user_ratings in ratings.groupby('userId'):
    n = len(user_ratings)
    n_test = max(1, int(n * 0.2))
    train_list.append(user_ratings.iloc[:-n_test])
    test_list.append(user_ratings.iloc[-n_test:])

train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

print(f'Train: {len(train_df):,} ratings ({len(train_df)/len(ratings)*100:.0f}%)')
print(f'Test:  {len(test_df):,} ratings ({len(test_df)/len(ratings)*100:.0f}%)')

## 3. Métricas de evaluación

Para comparar los modelos usamos métricas estándar de sistemas de recomendación:

- **Precision@K** — de las K películas recomendadas, ¿qué fracción le gustó realmente al usuario?
- **Recall@K** — de todas las películas que le gustaron, ¿qué fracción aparece en el top K?
- **NDCG@K** — mide la calidad del ranking, penalizando más los errores en las primeras posiciones

In [ ]:
def precision_at_k(recommended, relevant, k):
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / k if k > 0 else 0.0

def recall_at_k(recommended, relevant, k):
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / len(relevant) if relevant else 0.0

def ndcg_at_k(recommended, relevant, k):
    dcg  = sum(1/np.log2(i+2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1/np.log2(i+2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(user_recs, test_df, k=10):
    liked = test_df[test_df['rating'] >= 4]
    ps, rs, ns = [], [], []
    for uid, recs in user_recs.items():
        relevant = liked[liked['userId'] == uid]['movieId'].tolist()
        if not relevant:
            continue
        ps.append(precision_at_k(recs, relevant, k))
        rs.append(recall_at_k(recs, relevant, k))
        ns.append(ndcg_at_k(recs, relevant, k))
    return {
        f'Precision@{k}': round(np.mean(ps), 4),
        f'Recall@{k}':    round(np.mean(rs), 4),
        f'NDCG@{k}':      round(np.mean(ns), 4),
    }

K = 10
N_USERS = 200
sample_users = test_df['userId'].unique()[:N_USERS]
results = {}
print('Métricas definidas. Evaluando sobre', N_USERS, 'usuarios.')

## 4. Modelo 1 — Popularity Recommender

El modelo más simple: recomienda las películas con mayor rating medio a todos los usuarios por igual. No es personalizado, pero sirve como **baseline** para comparar con los demás.

In [ ]:
movie_stats = (
    train_df.groupby('movieId')
    .agg(avg_rating=('rating','mean'), count=('rating','count'))
    .reset_index()
)

top_popular = (
    movie_stats[movie_stats['count'] >= 50]
    .sort_values('avg_rating', ascending=False)
    .head(K)['movieId'].tolist()
)

print('Top 10 películas más populares:')
top_pop_titles = movies[movies['movieId'].isin(top_popular)][['title','movieId']]
for i, row in enumerate(top_popular, 1):
    title = movies[movies['movieId']==row]['title'].values[0]
    rating = movie_stats[movie_stats['movieId']==row]['avg_rating'].values[0]
    print(f'  {i}. {title} ({rating:.2f}⭐)')

user_recs_pop = {uid: top_popular for uid in sample_users}
results['Popularity'] = evaluate(user_recs_pop, test_df, K)
print('\nMétricas:', results['Popularity'])

## 5. Modelo 2 — Collaborative Filtering (SVD)

El **SVD (Singular Value Decomposition)** factoriza la matriz usuario-película en vectores latentes que capturan preferencias implícitas. Permite predecir el rating que daría un usuario a películas que no ha visto, basándose en el comportamiento de usuarios similares.

In [ ]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(train_df[['userId','movieId','rating']], reader)
trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

svd = SVD(n_factors=100, n_epochs=20, random_state=42)
svd.fit(trainset)

predictions = svd.test(testset)
print(f'RMSE del modelo SVD: {rmse(predictions):.4f}')
print('(Cuanto más bajo, mejor. Indica el error medio en la predicción de ratings)')

In [ ]:
user_recs_svd = {}

for uid in sample_users:
    watched = train_df[train_df['userId']==uid]['movieId'].tolist()
    candidates = movies[~movies['movieId'].isin(watched)]['movieId'].tolist()
    preds = [(mid, svd.predict(uid, mid).est) for mid in candidates]
    preds.sort(key=lambda x: x[1], reverse=True)
    user_recs_svd[uid] = [p[0] for p in preds[:K]]

results['Collaborative (SVD)'] = evaluate(user_recs_svd, test_df, K)
print('Métricas:', results['Collaborative (SVD)'])

## 6. Modelo 3 — Content-Based (TF-IDF)

Este modelo recomienda películas similares basándose en sus géneros. Los géneros se convierten en vectores TF-IDF y la similitud entre películas se calcula con la **similitud del coseno**. No aprende del comportamiento de los usuarios, sino de las características de las películas.

In [ ]:
cb_movies = movies.copy().reset_index(drop=True)
cb_movies['genres_text'] = cb_movies['genres_parsed'].apply(lambda g: ' '.join(g))

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(cb_movies['genres_text'])
sim_matrix = cosine_similarity(tfidf_matrix)

mid_to_idx = {row['movieId']: idx for idx, row in cb_movies.iterrows()}

print(f'Matriz de similitud: {sim_matrix.shape}')
print(f'Vocabulario TF-IDF: {len(tfidf.vocabulary_)} géneros únicos')
print('\nEjemplo — Películas similares a "Toy Story":')
idx = mid_to_idx.get(1)
if idx is not None:
    scores = list(enumerate(sim_matrix[idx]))
    scores.sort(key=lambda x: x[1], reverse=True)
    for i, (movie_idx, score) in enumerate(scores[1:6], 1):
        title = cb_movies.iloc[movie_idx]['title']
        print(f'  {i}. {title} (similitud: {score:.4f})')

In [ ]:
user_recs_cb = {}

for uid in sample_users:
    watched = train_df[train_df['userId']==uid]['movieId'].tolist()
    scores = np.zeros(len(cb_movies))
    for mid in watched:
        idx = mid_to_idx.get(mid)
        if idx is not None:
            scores += sim_matrix[idx]
    watched_idxs = [mid_to_idx[m] for m in watched if m in mid_to_idx]
    scores[watched_idxs] = -1
    top_idxs = np.argsort(scores)[::-1][:K]
    user_recs_cb[uid] = cb_movies.iloc[top_idxs]['movieId'].tolist()

results['Content-Based (TF-IDF)'] = evaluate(user_recs_cb, test_df, K)
print('Métricas:', results['Content-Based (TF-IDF)'])

## 7. Modelo 4 — Classification (Random Forest)

Modelo de **clasificación binaria**: predice si a un usuario le gustará una película (rating ≥ 4 = liked). Usa como features el perfil del usuario (edad, género, ocupación) y las características de la película (año, géneros en one-hot encoding).

In [ ]:
all_genres = sorted(set(g for genres in movies['genres_parsed'] for g in genres))
for genre in all_genres:
    movies[f'genre_{genre}'] = movies['genres_parsed'].apply(lambda g: 1 if genre in g else 0)

le = LabelEncoder()
users = users.copy()
users['gender_encoded'] = le.fit_transform(users['gender'])

genre_cols = [f'genre_{g}' for g in all_genres]
feature_cols = ['userId','movieId','year','age','gender_encoded','occupation'] + genre_cols

df_train = train_df.merge(movies[['movieId','year']+genre_cols], on='movieId')
df_train = df_train.merge(users[['userId','age','gender_encoded','occupation']], on='userId')
df_train['liked'] = (df_train['rating'] >= 4).astype(int)

X_train = df_train[feature_cols]
y_train = df_train['liked']

df_test = test_df.merge(movies[['movieId','year']+genre_cols], on='movieId')
df_test = df_test.merge(users[['userId','age','gender_encoded','occupation']], on='userId')
df_test['liked'] = (df_test['rating'] >= 4).astype(int)

X_test = df_test[feature_cols]
y_test = df_test['liked']

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Features: {len(feature_cols)}')
print(f'Balance clases — Liked: {y_train.sum():,} | Not liked: {(y_train==0).sum():,}')

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print('=== MÉTRICAS DEL CLASIFICADOR ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not liked', 'Liked']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Not liked', 'Liked'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusión — Random Forest', fontsize=13, fontweight='bold')

# Feature importance (top 15)
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top_features = importances.sort_values(ascending=False).head(15)
axes[1].barh(top_features.index, top_features.values, color='steelblue')
axes[1].set_title('Top 15 Features más Importantes', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importancia')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
user_recs_rf = {}

for uid in sample_users:
    user = users[users['userId']==uid]
    if user.empty: continue
    user = user.iloc[0]
    watched = train_df[train_df['userId']==uid]['movieId'].tolist()
    candidates = movies[~movies['movieId'].isin(watched)].copy()
    candidates['userId'] = uid
    candidates['age'] = user['age']
    candidates['gender_encoded'] = user['gender_encoded']
    candidates['occupation'] = user['occupation']
    probs = rf.predict_proba(candidates[feature_cols])[:, 1]
    candidates = candidates.copy()
    candidates['prob'] = probs
    user_recs_rf[uid] = candidates.sort_values('prob', ascending=False).head(K)['movieId'].tolist()

results['Classification (RF)'] = evaluate(user_recs_rf, test_df, K)
print('Métricas:', results['Classification (RF)'])

## 8. Comparativa de modelos

In [ ]:
results_df = pd.DataFrame(results).T
results_df.columns = [f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']
print('=== COMPARATIVA DE MODELOS ===')
display(results_df.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#607D8B', '#2196F3', '#9C27B0', '#FF5722']
metrics = [f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']

for ax, metric in zip(axes, metrics):
    vals = results_df[metric]
    bars = ax.bar(vals.index, vals.values, color=colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, vals.max() * 1.25)
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle(f'Comparativa de Modelos — Top {K} Recomendaciones',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Conclusiones

**Collaborative Filtering (SVD)** suele ser el modelo más preciso en sistemas de recomendación con suficientes datos, ya que captura patrones latentes de comportamiento entre usuarios. Su principal limitación es el **cold start**: no puede recomendar a usuarios nuevos sin historial.

**Content-Based (TF-IDF)** no depende del comportamiento de otros usuarios, por lo que no tiene el problema de cold start. Sin embargo, al usar solo géneros como features, sus recomendaciones son menos precisas que las del SVD.

**Classification (Random Forest)** aporta un enfoque diferente al reformular el problema como clasificación binaria. Incorpora features del perfil del usuario (edad, género, ocupación) que los otros modelos ignoran, lo que puede ser ventajoso en datasets con perfiles ricos.

**Popularity** actúa como baseline y, aunque no es personalizado, es útil para usuarios nuevos sin historial.